<a href="https://colab.research.google.com/github/SPLV2024/MLMODEL/blob/main/Untitled7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas openpyxl gensim


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip


--2024-07-26 15:15:23--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2024-07-26 15:15:23--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2024-07-26 15:15:24--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [6]:
import pandas as pd
from gensim.models import KeyedVectors

def load_workbook(file_path):
    """Load the Excel workbook and return a dictionary of DataFrames keyed by sheet names."""
    excel_data = pd.read_excel(file_path, sheet_name=None)
    return excel_data

def load_word_embeddings(embedding_file):
    """Load pre-trained word embeddings."""
    model = KeyedVectors.load_word2vec_format(embedding_file, binary=False, no_header=True)
    return model

def get_similarity_score(description, text, model):
    """Calculate the similarity score between the description and text using word embeddings."""
    description_words = [word for word in description.split() if word in model]
    text_words = [word for word in text.split() if word in model]

    if not description_words or not text_words:
        return 0

    similarity = model.n_similarity(description_words, text_words)
    return similarity

def search_data(dictionary, description, model, threshold=0.5):
    """
    Search for table names and column descriptions within the workbook dictionary based on a description.

    Args:
        dictionary (dict): Dictionary of DataFrames representing the workbook.
        description (str): Description provided by the client.
        model: Pre-trained word embeddings model.
        threshold (float): Similarity threshold for filtering results.

    Returns:
        pd.DataFrame: DataFrame with columns: Sheet, Table Name, Column Name, Description.
    """
    results = []
    for sheet, df in dictionary.items():
        df = df.astype(str)
        for idx, row in df.iterrows():
            row_text = ' '.join(row.values)
            similarity = get_similarity_score(description, row_text, model)
            if similarity >= threshold:
                results.append({
                    'Sheet': sheet,
                    'Table Name': row.get('TOPIC', 'N/A'),
                    'Column Name': row.get('SUB TOPICS COVERED ', 'N/A'),
                    'Description': row.get('Description', 'N/A')
                })

    return pd.DataFrame(results)



In [4]:
def display_results(results):
    """Display the search results."""
    print(results)

# Path to your Excel file
file_path = '/content/drive/MyDrive/Course Structure for Analytics Boot camp.xlsx'

# Load the workbook
workbook_data = load_workbook(file_path)

# Load pre-trained word embeddings
embedding_file = 'glove.6B.50d.txt'  # Replace with the path to your embeddings file
word_embeddings_model = load_word_embeddings(embedding_file)

# Description provided by the client
description = 'Cleansing, Transforming and Shaping Data'  # Replace with the description provided by the client

# Get search results
search_results = search_data(workbook_data, description, word_embeddings_model)

# Display the results
display_results(search_results)


NameError: name 'load_workbook' is not defined

In [9]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def load_workbook(file_path):
    """Load the Excel workbook and return a dictionary of DataFrames keyed by sheet names."""
    excel_data = pd.read_excel(file_path, sheet_name=None)
    return excel_data

def prepare_corpus(dictionary):
    """Prepare a corpus from the workbook dictionary."""
    corpus = []
    row_metadata = []
    for sheet, df in dictionary.items():
        df = df.astype(str)
        for idx, row in df.iterrows():
            row_text = ' '.join(row.values)
            corpus.append(row_text)
            row_metadata.append({
                'Sheet': sheet,
                'Table Name': row.get('S.No', 'N/A'),
                'Column Name': row.get('TOPIC', 'N/A'),
                'Description': row.get('SUB TOPICS COVERED', 'N/A')
            })
    return corpus, row_metadata

def search_data(description, corpus, row_metadata, vectorizer, threshold=0.2):
    """
    Search for relevant rows based on a description using TF-IDF and cosine similarity.

    Args:
        description (str): Description provided by the client.
        corpus (list): List of documents.
        row_metadata (list): Metadata for each row.
        vectorizer: TF-IDF vectorizer.
        threshold (float): Similarity threshold for filtering results.

    Returns:
        pd.DataFrame: DataFrame with columns: Sheet, Table Name, Column Name, Description.
    """
    description_vec = vectorizer.transform([description])
    corpus_vec = vectorizer.transform(corpus)
    similarities = cosine_similarity(description_vec, corpus_vec).flatten()

    results = []
    for idx, similarity in enumerate(similarities):
        if similarity >= threshold:
            results.append(row_metadata[idx])

    return pd.DataFrame(results)

def display_results(results):
    """Display the search results."""
    print(results)

# Path to your Excel file
file_path = '/content/drive/MyDrive/Course Structure for Analytics Boot camp.xlsx'

# Load the workbook
workbook_data = load_workbook(file_path)

# Prepare the corpus and metadata
corpus, row_metadata = prepare_corpus(workbook_data)

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer()
vectorizer.fit(corpus)

# Description provided by the client
description = 'tables managing column soring '  # Replace with the description provided by the client

# Get search results
search_results = search_data(description, corpus, row_metadata, vectorizer)

# Display the results
display_results(search_results)


                Sheet Table Name             Column Name Description
0                 SQL        N/A                     N/A         N/A
1                 SQL        N/A                     N/A         N/A
2                 SQL        N/A                     N/A         N/A
3                 SQL        N/A                     N/A         N/A
4  Data Visualization        nan                     nan         N/A
5  Data Visualization        nan                     nan         N/A
6  Data Visualization        3.0  Designing a Data Model         N/A


In [12]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download stopwords if not already downloaded
nltk.download('stopwords')

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and numbers
    text = re.sub(r'\W+', ' ', text)
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

def find_similar_rows(input_text, df, X, vectorizer):
    input_processed = preprocess_text(input_text)
    input_vector = vectorizer.transform([input_processed])
    similarities = cosine_similarity(input_vector, X)
    df['similarity'] = similarities[0]
    # Return the rows with high similarity scores
    return df[df['similarity'] > 0.1].sort_values(by='similarity', ascending=False)

# Load the dataset
df = pd.read_excel('/content/drive/MyDrive/Course Structure for Analytics Boot camp.xlsx')

df

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Course Structure for Business Analytics & AI,Unnamed: 1
0,S.NO,DESCRIPTION
1,1,NaN
2,2,Excel Foundation
3,3,SQL Programming
4,4,Python Programming
5,5,Basic Statistics
6,6,Business Data Visualization - Power BI
7,7,Introduction to BA and AI (Masterclass)
8,8,Business Analytics visualization -Tableau Trai...
9,9,Big Data Application and Practices


In [24]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download stopwords if not already downloaded
nltk.download('stopwords')

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and numbers
    text = re.sub(r'\W+', ' ', text)
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

def find_similar_rows(input_text, df, X, vectorizer):
    input_processed = preprocess_text(input_text)
    input_vector = vectorizer.transform([input_processed])
    similarities = cosine_similarity(input_vector, X)
    df['similarity'] = similarities[0]
    # Return the rows with high similarity scores
    return df[df['similarity'] > 0.1].sort_values(by='similarity', ascending=False)

def process_sheet(sheet_name, df, input_text):
    df['combined_text'] = df['TOPIC'].astype(str) + ' ' + df['SUB TOPICS COVERED'].astype(str)
    df['processed_text'] = df['combined_text'].apply(preprocess_text)
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(df['processed_text'])
    matching_rows = find_similar_rows(input_text, df, X, vectorizer)
    return matching_rows[['TOPIC', 'SUB TOPICS COVERED']]

# Load the Excel file
excel_file = pd.ExcelFile('/content/drive/MyDrive/Course Structure for Analytics Boot camp.xlsx')

input_text = "bar chart in tableau"

all_matching_rows = pd.DataFrame(columns=['TOPIC', 'SUB TOPICS COVERED'])

# Iterate through each sheet
for sheet_name in excel_file.sheet_names:
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
    if {'TOPIC', 'SUB TOPICS COVERED'}.issubset(df.columns):
        matching_rows = process_sheet(sheet_name, df, input_text)
        all_matching_rows = pd.concat([all_matching_rows, matching_rows])

all_matching_rows


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,TOPIC,SUB TOPICS COVERED


In [42]:
import pandas as pd
import pickle

# Load all sheets from the Excel file
file_path = 'requirements.xlsx'
sheets = pd.read_excel(file_path, sheet_name=None)

# Preprocess each sheet and store in a dictionary
data_dict = {}
for sheet_name, df in sheets.items():
    # Normalize column names to lowercase and strip any leading/trailing whitespace
    df.columns = [col.lower().strip() for col in df.columns]
    # Convert all records to lowercase
    df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    # Convert to list of dictionaries
    data_dict[sheet_name.lower().strip()] = df.to_dict(orient='records')

# Save the processed data to a file for quick loading later
with open('processed_data.pkl', 'wb') as f:
    pickle.dump(data_dict, f)

Data preprocessing and saving completed.


In [52]:
!pip install rapidfuzz


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 28.4 MB/s eta 0:00:00


In [57]:
import pickle
import pandas as pd
from rapidfuzz import fuzz

# Load the processed data from the file
with open('processed_data.pkl', 'rb') as f:
    data_dict = pickle.load(f)

def find_relevant_sheet_and_extract(prompt):
    prompt = prompt.lower()  # Normalize the prompt to lowercase

    best_match = None
    highest_score = 0
    matching_rows = pd.DataFrame()

    for sheet_name, data in data_dict.items():
        df = pd.DataFrame(data)
        if 'topic' in df.columns and 'sub topics covered' in df.columns:
            df['match_score'] = df.apply(
                lambda row: max(fuzz.partial_ratio(prompt, str(row['topic']).lower()),
                                fuzz.partial_ratio(prompt, str(row['sub topics covered']).lower())), axis=1
            )
            score = df['match_score'].max()

            if score > highest_score:
                highest_score = score
                best_match = sheet_name
                matching_rows = df[df['match_score'] == score]

    if best_match:
        if 'topic' in matching_rows.columns and 'sub topics covered' in matching_rows.columns:
            # Display the matching rows with the topic and sub topics covered columns
            result_df = matching_rows[['topic', 'sub topics covered']].reset_index(drop=True)
            return best_match, result_df
        else:
            return best_match, pd.DataFrame(columns=['topic', 'sub topics covered'])  # Return an empty DataFrame with the correct columns
    else:
        return None, pd.DataFrame(columns=['topic', 'sub topics covered'])  # Return an empty DataFrame with the correct columns

# Example usage
prompt = input("Enter your prompt: ")
sheet_name, result = find_relevant_sheet_and_extract(prompt)
if not result.empty:
    print(f"Best matching sheet: {sheet_name}")
    print("Matching columns and rows:")
    print(result)
else:
    print("No matching rows found.")
    print(result)


Enter your prompt: dash
Best matching sheet: data visualization
Matching columns and rows:
  topic   sub topics covered
0   NaN  Creating Dashboards


In [49]:
# Example usage
while True:
    prompt = input("Enter your prompt: ")
    if prompt.lower() == 'exit':
        break
    sheet_name, result = find_relevant_sheet_and_extract(prompt)
    if isinstance(result, pd.DataFrame):
        print(f"Best matching sheet: {sheet_name}")
        print(result)
    else:
        print(result)


Enter your prompt: tableau barchart 
No matching sheet found.


KeyboardInterrupt: Interrupted by user